# SECTION-05. Customer Analysis

## Business Problem

Analyze customer purchasing behavior to identify high-value customers and understand customer buying patterns.

In [ ]:
--Before going to perform the analysis, make sure to select the database in which you want to run the query.
/*Requied tables:
        sales.SalesOrderHeader 
        person.person
        sales.customer
*/


Commands completed successfully.

Total execution time: 00:00:00.001

In [ ]:
set nocount on;
--query.1
print 'Which customers have generated the highest lifetime revenue and average value  for Adventure Works..?';


with lifetime_revenue as
(
    select
        c.customerid,
        c.personid,
        count(h.salesorderid) as total_orders,
        sum(h.totaldue) as total_revenue,
        avg(h.totaldue) as average_order_value
    from sales.customer c
    join sales.salesorderheader h
        on c.customerid = h.customerid
    group by
        c.customerid,
        c.personid
)

select top (10)
    l.customerid,
    p.firstname + ' ' + p.lastname as customer_name,
    l.total_orders,
    l.total_revenue,
    l.average_order_value
from lifetime_revenue l
join person.person p
    on l.personid = p.businessentityid
order by
    l.total_revenue desc;
go




# SECTION 06: Product Analysis

## Objective

 analyze product sales performance by identifying the best-selling products, evaluating revenue generated by each product category, and determining the top-performing product within every category to support inventory management and business decision-making.

In [ ]:
--Before going to perform the analysis, make sure to select the database in which you want to run the query.
/*Requied tables:
        Sales.SalesOrderDetail
        Production.Product
        Production.ProductSubcategory
        Production.ProductCategory
*/

In [46]:
set nocount on;
--query.1
print 'Best-Selling Products';

with best_selling_product as
(
    select
        p.productid,
        p.name,
        sum(o.orderqty) as total_quantity_sold,
        sum(o.linetotal) as total_revenue
    from sales.salesorderdetail o
    join production.product p
        on o.productid = p.productid
    group by
        p.productid,
        p.name
)
select top (10)
    productid,
    name,
    total_quantity_sold,
    total_revenue
from best_selling_product
order by
    total_revenue desc;
go

--query.2
print 'Total Revenue and Quantity Sold by Product Category';

with productbycategory as
(
    select
        pc.productcategoryid,
        pc.name as category_name,
        ps.productsubcategoryid
    from production.productcategory pc
    join production.productsubcategory ps
        on pc.productcategoryid = ps.productcategoryid
),

high_revenue_products as
(
    select
        p.productid,
        p.name as product_name,
        p.productsubcategoryid,
        sum(o.orderqty) as total_quantity_sold,
        sum(o.linetotal) as total_revenue
    from sales.salesorderdetail o
    join production.product p
        on o.productid = p.productid
    group by
        p.productid,
        p.name,
        p.productsubcategoryid
)

select
    pc.category_name,
    sum(h.total_quantity_sold) as total_quantity_sold,
    sum(h.total_revenue) as total_revenue
from high_revenue_products h
join productbycategory pc
    on h.productsubcategoryid = pc.productsubcategoryid
group by
    pc.category_name
order by
    total_revenue desc;
go

--querry.3

print 'Top-Selling Product by Category';

with productbycategory as
(
    select
        pc.productcategoryid,
        pc.name as category_name,
        ps.productsubcategoryid
    from production.productcategory pc
    join production.productsubcategory ps
        on pc.productcategoryid = ps.productcategoryid
),

high_revenue_products as
(
    select
        p.productid,
        p.name as product_name,
        p.productsubcategoryid,
        sum(o.orderqty) as total_quantity_sold,
        sum(o.linetotal) as total_revenue
    from sales.salesorderdetail o
    join production.product p
        on o.productid = p.productid
    group by
        p.productid,
        p.name,
        p.productsubcategoryid
),

ranked_products as
(
    select
        pc.category_name,
        h.product_name,
        h.total_quantity_sold,
        h.total_revenue,
        dense_rank() over (
            partition by pc.category_name
            order by h.total_revenue desc
        ) as rank_within_category
    from high_revenue_products h
    join productbycategory pc
        on h.productsubcategoryid = pc.productsubcategoryid
)

select *
from ranked_products
where rank_within_category = 1;
go

Best-Selling Products

productid | name                    | total_quantity_sold | total_revenue 
----------+-------------------------+---------------------+---------------
782       | Mountain-200 Black, 38  | 2977                | 4400592.800400
783       | Mountain-200 Black, 42  | 2664                | 4009494.761841
779       | Mountain-200 Silver, 38 | 2394                | 3693678.025272
780       | Mountain-200 Silver, 42 | 2234                | 3438478.860423
781       | Mountain-200 Silver, 46 | 2216                | 3434256.941928
784       | Mountain-200 Black, 46  | 2111                | 3309673.216908
793       | Road-250 Black, 44      | 1642                | 2516857.314918
794       | Road-250 Black, 48      | 1498                | 2347655.953454
795       | Road-250 Black, 52      | 1245                | 2012447.775000
753       | Road-150 Red, 56        | 664                 | 1847818.628000
(10 rows)

Total Revenue and Quantity Sold by Product Category

category_name

# SECTION 07: Employee & Territory Analysis

## Objective

In this section, we will analyze the performance of sales employees and sales territories to evaluate workforce productivity, identify high-performing regions, and support data-driven sales management and resource allocation.

### Business Scenario
The sales director wants to evaluate the performance of each salesperson based on customer orders, sales volume, and revenue generated. This analysis helps recognize top performers, optimize sales strategies, and support employee performance reviews.

In [47]:
--Before going to perform the analysis, make sure to select the database in which you want to run the query.
/*Required tables:
    Sales.SalesOrderHeader
    Sales.SalesPerson
    HumanResources.Employee
    Person.Person
*/

Commands completed successfully.

Total execution time: 00:00:00.000

In [ ]:
--querry.1

print 'Top 10 Salespersons by Performance:';

with salesperson_performance as
(
    select
        soh.salespersonid,
        p.firstname + ' ' + p.lastname as salesperson_name,
        count(soh.salesorderid) as total_orders,
        count(distinct soh.customerid) as total_customers,
        sum(soh.totaldue) as total_sales,
        avg(soh.totaldue) as average_order_value
    from sales.salesorderheader soh
    join person.person p
        on soh.salespersonid = p.businessentityid
    where soh.salespersonid is not null
    group by
        soh.salespersonid,
        p.firstname,
        p.lastname
)

select top 10
    salespersonid,
    salesperson_name,
    total_orders,
    total_customers,
    total_sales,
    average_order_value
from salesperson_performance
order by total_sales desc;
go


--querry.2

print 'Salesperson Performance by Territory:';

with salesperson_territory_performance as
(
    select 
           t.name as country,
           t.countryregioncode as region,
           count(h.salesorderid) as total_orders,
           count(DISTINCT h.customerid) as total_customers,
           sum(h.totaldue) as total_sales,
           avg(h.totaldue) as average_order_value
    from sales.salesterritory t
    join sales.salesorderheader h
        on t.territoryid = h.territoryid
    group by 
        t.name,
        t.countryregioncode
)
select country, region, total_orders, total_customers, total_sales, average_order_value
from salesperson_territory_performance
order by total_sales desc,average_order_value desc;



Top 10 Salespersons by Performance:

salespersonid | salesperson_name         | total_orders | total_customers | total_sales   | average_order_value
--------------+--------------------------+--------------+-----------------+---------------+--------------------
276           | Linda Mitchell           | 418          | 69              | 11695019.0605 | 27978.5144         
277           | Jillian Carson           | 473          | 121             | 11342385.8968 | 23979.6742         
275           | Michael Blythe           | 450          | 118             | 10475367.0751 | 23278.5935         
289           | Jae Pak                  | 348          | 62              | 9585124.9477  | 27543.4624         
279           | Tsvi Reiter              | 429          | 74              | 8086073.6761  | 18848.6565         
281           | Shu Ito                  | 242          | 35              | 7259567.8761  | 29998.2143         
282           | José Saraiva             | 271          | 67       